# RoomBeacon — Post-Silver Processing and Feature Contract

This notebook consumes canonical Silver, performs post-Silver analytical feature work, and publishes a safe modeling contract. It does **not** create train/validation/test partitions, select models, or persist a redundant modeling dataset. Notebook 04 exclusively owns group-aware chronological splitting and model evaluation.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'crawler').is_dir() and (p/'analytics').is_dir())
sys.path.insert(0,str(PROJECT_ROOT))
import numpy as np,pandas as pd,duckdb
from dotenv import load_dotenv
from IPython.display import display,Markdown
load_dotenv(PROJECT_ROOT/'.env',override=False)
from analytics.duckdb.connection import resolve_runtime_path
from roombeacon_crawler.config.get_env import env
from notebooks.utils.location_analysis import haversine_distance_km
from notebooks.utils.modeling_benchmark import TARGET,FORBIDDEN_PREDICTORS,audit_features,build_feature_sets,build_modeling_eligibility,eligibility_funnel
pd.set_option('display.max_columns',30); SEED=42

## 01 Load Canonical Silver

In [2]:
SILVER_PATH=resolve_runtime_path(env.processing.silver_dir)/'rental_listings.parquet'
assert SILVER_PATH.exists(),f'Canonical Silver not found: {SILVER_PATH}'
with duckdb.connect(':memory:') as con: silver_df=con.execute('select * from read_parquet(?)',[str(SILVER_PATH)]).df()
assert len(silver_df)>0 and silver_df.rental_post_id.notna().all() and silver_df.rental_post_id.is_unique
display(pd.Series({'path':str(SILVER_PATH),'rows':len(silver_df),'columns':len(silver_df.columns)},name='value').to_frame())

,value
path,/data/projects/roombeacon/source/roombeacon-sy...
rows,132436
columns,80


## 02 Analytical eligibility and diagnostic-only variables

In [3]:
processed_df=silver_df.copy()
processing_masks=build_modeling_eligibility(processed_df,policy='PERMISSIVE')
processed_df['analytical_price_eligible']=processing_masks.semantic_target_supported
processed_df['analytical_area_eligible']=processing_masks.area_supported
processed_df['model_base_eligible']=processing_masks.final_eligible
price=pd.to_numeric(processed_df[TARGET],errors='coerce'); area=pd.to_numeric(processed_df.area_value_clean,errors='coerce')
processed_df['price_per_area_analysis']=price/area.where(area.gt(0))
processed_df.loc[~np.isfinite(processed_df.price_per_area_analysis),'price_per_area_analysis']=np.nan
display(processed_df[['analytical_price_eligible','analytical_area_eligible','model_base_eligible']].sum().to_frame('rows'))
display(eligibility_funnel(processing_masks))
display(processed_df.price_per_area_analysis.describe().to_frame())
display(Markdown('`price_per_area_analysis` is **TARGET-DERIVED / ANALYSIS-ONLY** and is programmatically forbidden from every modeling feature set.'))

,rows
analytical_price_eligible,114436
analytical_area_eligible,112226
model_base_eligible,112226


,Stage,Rows,Removed / reviewed,Percent of Silver,Policy
0,Silver rows,132436,0,100.000000,PERMISSIVE
1,Numeric target candidate,128051,4385,96.688967,PERMISSIVE
2,Lineage trusted,118886,9165,89.768643,PERMISSIVE
3,Semantic target supported,114436,4450,86.408529,PERMISSIVE
4,Area supported,112226,2210,84.739799,PERMISSIVE
5,Rental-compatible,112226,0,84.739799,PERMISSIVE
6,Final model eligible,112226,0,84.739799,PERMISSIVE


,price_per_area_analysis
count,1.171690e+05
mean,1.706864e+05
std,3.354407e+05
min,9.090909e-02
25%,1.150000e+05
50%,1.500000e+05
75%,1.933333e+05
max,9.166667e+07


`price_per_area_analysis` is **TARGET-DERIVED / ANALYSIS-ONLY** and is programmatically forbidden from every modeling feature set.

## 03 Optional trusted spatial diagnostic

In [4]:
REFERENCE_LOCATION=None
processed_df['distance_km_analysis']=np.nan
if REFERENCE_LOCATION is not None:
 required={'latitude','longitude'}
 if not required<=set(REFERENCE_LOCATION): raise ValueError('REFERENCE_LOCATION requires latitude and longitude')
 trusted=processed_df.has_trusted_coordinate.fillna(False)
 processed_df.loc[trusted,'distance_km_analysis']=haversine_distance_km(processed_df.loc[trusted,'map_latitude'],processed_df.loc[trusted,'map_longitude'],REFERENCE_LOCATION['latitude'],REFERENCE_LOCATION['longitude'])
display(pd.Series({'trusted_coordinate_rows':int(processed_df.has_trusted_coordinate.sum()),'distance_rows':int(processed_df.distance_km_analysis.notna().sum())},name='rows').to_frame())

,rows
trusted_coordinate_rows,2277
distance_rows,0


## 04 Safe feature contract

In [5]:
contract_frame=processed_df.assign(title_length=processed_df.title_clean.astype('string').str.len().astype('Float64'))
SAFE_FEATURE_SETS=build_feature_sets(contract_frame.columns)
for name,features in SAFE_FEATURE_SETS.items(): audit_features(features)
for forbidden in ['price_per_area_analysis',TARGET,'price_amount','rental_post_id','duplicate_candidate_group']:
 try: audit_features(['area_value_clean',forbidden])
 except ValueError: pass
 else: raise AssertionError(f'Leakage guard failed for {forbidden}')
feature_contract=pd.DataFrame([{'Feature Set':name,'Safe Predictors':', '.join(features)} for name,features in SAFE_FEATURE_SETS.items()])
excluded=sorted(FORBIDDEN_PREDICTORS|{'all price_* status/evidence fields','parser/quality fields that encode target validation'})
contract_summary=pd.DataFrame([
 {'Contract Item':'Target','Fields':TARGET},
 {'Contract Item':'Safe candidates','Fields':', '.join(sorted(set(sum(SAFE_FEATURE_SETS.values(),[]))))},
 {'Contract Item':'Grouping key source','Fields':'duplicate_candidate_group, falling back to rental_post_id in Notebook 04'},
 {'Contract Item':'Chronological time','Fields':'latest_observed_at'},
 {'Contract Item':'Analysis-only','Fields':'price_per_area_analysis, distance_km_analysis'},
 {'Contract Item':'Eligibility/audit','Fields':'listing_intent, rental_scope, price_target_trust_status, price_model_suitability, area_model_suitability, admin_consistency_status; not predictors'},
 {'Contract Item':'Excluded/leakage','Fields':', '.join(excluded)},
 {'Contract Item':'Split owner','Fields':'Notebook 04 only'},
])
display(feature_contract); display(contract_summary)
assert 'dataset_split' not in processed_df.columns

,Feature Set,Safe Predictors
0,F1 — AREA ONLY,area_value_clean
1,F2 — AREA + SOURCE,"area_value_clean, source_code"
2,F3 — AREA + LOCATION,"area_value_clean, ward_current, district_text_..."
3,F4 — AREA + SOURCE + LOCATION,"area_value_clean, source_code, ward_current, d..."
4,F5 — FULL SAFE TABULAR,"area_value_clean, source_code, ward_current, d..."


,Contract Item,Fields
0,Target,price_model_value
1,Safe candidates,"area_value_clean, district_text_extracted, pro..."
2,Grouping key source,"duplicate_candidate_group, falling back to ren..."
3,Chronological time,latest_observed_at
4,Analysis-only,"price_per_area_analysis, distance_km_analysis"
5,Eligibility/audit,"listing_intent, rental_scope, price_target_tru..."
6,Excluded/leakage,"all price_* status/evidence fields, duplicate_..."
7,Split owner,Notebook 04 only


## 05 Processing Summary

In [6]:
summary=pd.Series({'silver_input_rows':len(silver_df),'analytical_rows':len(processed_df),'model_base_eligible_rows':int(processed_df.model_base_eligible.sum()),'safe_feature_sets':len(SAFE_FEATURE_SETS),'legacy_split_columns':int('dataset_split' in processed_df.columns),'persisted_redundant_model_dataset':False},name='value')
assert len(processed_df)==len(silver_df); display(summary.to_frame()); display(Markdown('**Notebook 03 complete. Modeling split, selection, tuning, locking, and TEST evaluation remain exclusively in Notebook 04.**'))

,value
silver_input_rows,132436
analytical_rows,132436
model_base_eligible_rows,112226
safe_feature_sets,5
legacy_split_columns,0
persisted_redundant_model_dataset,False


**Notebook 03 complete. Modeling split, selection, tuning, locking, and TEST evaluation remain exclusively in Notebook 04.**